# Polynomial Regression: SVD Optimizer Analysis (Full Scan, 10 seeds)

Reproduces the 'Custom plots' section of `polynomial_analysis.ipynb` but averages over 10 model seeds per config, showing mean curves with ±1 std bands. Best configs selected by lowest *mean* final val loss.

## 1. Setup & Data Loading

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from style import set_style, lr_labels, load_results, average_over_seeds
set_style()

PLOT_DIR = Path('plots/polynomial_fullscan')
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {PLOT_DIR.resolve()}")

In [ ]:
df = load_results("polynomial_scan")
if 'run_id' in df.columns:
    df = df.drop(columns=['run_id'])
print(f"Total: {len(df)} runs, optimizers: {sorted(df['optimizer'].unique())}")
print(f"Model seeds: {sorted(df['model_seed'].unique())}")
print(f"Batch sizes: {sorted(df['batch_size'].unique())}")

In [ ]:
# Remove incomplete LBFGS runs (NaN at end of val curve)
def is_bad(row):
    if row['optimizer'] == 'LBFGS' and np.isnan(row['losses']['val'][-1]):
        return True
    return False
df = df[~df.apply(is_bad, axis=1)].reset_index(drop=True)
print(f"After removing bad LBFGS runs: {len(df)}")

def get_final_loss(row, loss_type='val'):
    losses = row['losses'][loss_type]
    for val in reversed(losses):
        if val is not None and not (isinstance(val, float) and np.isnan(val)):
            return val
    return np.nan

df['final_val_loss'] = df.apply(lambda r: get_final_loss(r, 'val'), axis=1)
df['final_train_loss'] = df.apply(lambda r: get_final_loss(r, 'train'), axis=1)
df['total_time'] = df['losses'].apply(lambda l: l.get('total_time', np.nan))
df['avg_epoch_time'] = df['losses'].apply(lambda l: l.get('avg_epoch_time', np.nan))

In [ ]:
df_avg = average_over_seeds(df, seed_col='model_seed')
print(f"After averaging: {len(df_avg)} configs; n_seeds per config: {sorted(df_avg['n_seeds'].unique())}")

df_svd = df_avg[df_avg['optimizer'] == 'SVD'].copy()
df_baseline = df_avg[df_avg['optimizer'] != 'SVD'].copy()

bs = sorted(df_avg['batch_size'].unique())[0]
baseline_optimizers = sorted(df_baseline['optimizer'].unique().tolist())
k_fractions = sorted(df_svd['k_fraction'].dropna().unique())
svd_lrs = sorted(df_svd['lr'].dropna().unique())
svd_rtols = sorted(df_svd['rtol'].dropna().unique())

print(f"SVD: {len(df_svd)} configs; k_fractions={k_fractions}; lrs={svd_lrs}; rtols={svd_rtols}")
print(f"Baseline: {len(df_baseline)} configs; optimizers={baseline_optimizers}")

In [ ]:
# Helpers for mean/std curve extraction and plotting with shaded bands.
def mean_std(row, key, container='losses'):
    """Return (mean, std) arrays for a key inside `container` dict (losses or svd_info)."""
    src = row[container]
    std_src = row.get(f'{container}_std')
    mean = np.asarray(src[key], dtype=float)
    if isinstance(std_src, dict) and std_src.get(key) is not None:
        std = np.asarray(std_src[key], dtype=float)
    else:
        std = np.zeros_like(mean)
    n = min(len(mean), len(std))
    return mean[:n], std[:n]

def plot_band(ax, x, mean, std, color, label=None, lw=2, zorder=1, alpha=0.22, ls='-'):
    x = np.asarray(x)[:len(mean)]
    ax.plot(x, mean, color=color, linewidth=lw, label=label, zorder=zorder, linestyle=ls)
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=alpha, linewidth=0, zorder=zorder - 0.1)

def best_row(sub):
    return sub.loc[sub['final_val_loss'].idxmin()]

## 2. Best Performance Summary

In [ ]:
rows = []
b = best_row(df_svd)
rows.append({'optimizer': 'SVD', 'final_val_loss': b['final_val_loss'], 'final_val_loss_std': b['final_val_loss_std'],
             'lr': b['lr'], 'k': b['k'], 'rtol': b['rtol'], 'total_time': b['total_time']})
for opt in baseline_optimizers:
    sub = df_baseline[df_baseline['optimizer'] == opt]
    b = best_row(sub)
    rows.append({'optimizer': opt, 'final_val_loss': b['final_val_loss'], 'final_val_loss_std': b['final_val_loss_std'],
                 'lr': b['lr'], 'k': None, 'rtol': None, 'total_time': b['total_time']})
best_df = pd.DataFrame(rows)
print("Best configs by mean final val loss:")
print(best_df.to_string(index=False))

## 3. Custom plots

### Train loss, best of each optimizer

In [ ]:
colors = sns.color_palette("deep")
fig, ax = plt.subplots(figsize=(8, 6))

best_svd_row = best_row(df_svd[df_svd['batch_size'] == bs])
train_mean, train_std = mean_std(best_svd_row, 'train')
n_epochs = len(train_mean)
epochs_train = np.arange(1, n_epochs + 1)
plot_band(ax, epochs_train, train_mean, train_std, color='k', lw=3, zorder=3,
          label=f"Sven ($\\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)")

for i, opt in enumerate(['SGD', 'PolyakSGD', 'RMSprop', 'Adam', 'LBFGS']):
    sub = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    if len(sub) == 0:
        continue
    b = best_row(sub)
    m, s = mean_std(b, 'train')
    plot_band(ax, epochs_train, m, s, color=colors[i], lw=2, zorder=2, label=opt)

ax.set_xlabel('Epoch')
ax.set_ylabel('Train Loss')
ax.set_yscale('log')
plt.legend(ncol=2, loc='upper right', frameon=True, framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None, 5])
plt.xlim(0, n_epochs + 1)
plt.xticks(np.arange(0, n_epochs + 1, max(1, n_epochs // 10)))
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'train_loss_best.pdf')
plt.show()

### Val loss, best of each optimizer

In [ ]:
colors = sns.color_palette("deep")
fig, ax = plt.subplots(figsize=(8, 6))

best_svd_row = best_row(df_svd[df_svd['batch_size'] == bs])
val_mean, val_std = mean_std(best_svd_row, 'val')
n_val = len(val_mean)
epochs_val = np.arange(n_val)
plot_band(ax, epochs_val, val_mean, val_std, color='k', lw=3, zorder=3,
          label=f"Sven ($\\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)")

for i, opt in enumerate(['SGD', 'PolyakSGD', 'RMSprop', 'Adam', 'LBFGS']):
    sub = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    if len(sub) == 0:
        continue
    b = best_row(sub)
    m, s = mean_std(b, 'val')
    plot_band(ax, epochs_val, m, s, color=colors[i], lw=2, zorder=2, label=opt)

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
plt.legend(ncol=2, loc='upper right', frameon=True, framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None, 2])
plt.xlim(0, n_val)
plt.xticks(np.arange(0, n_val + 1, max(1, n_val // 10)))
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'val_loss_best.pdf')
plt.show()

### Wall time vs train loss

In [ ]:
colors = sns.color_palette("deep")
fig, ax = plt.subplots(figsize=(8, 6))

best_svd_row = best_row(df_svd[df_svd['batch_size'] == bs])
train_mean, train_std = mean_std(best_svd_row, 'train')
et_mean, _ = mean_std(best_svd_row, 'epoch_times')
t = np.cumsum(et_mean[:len(train_mean)])
plot_band(ax, t, train_mean, train_std, color='k', lw=3, zorder=3,
          label=f"Sven ($\\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)")

for i, opt in enumerate(['SGD', 'PolyakSGD', 'RMSprop', 'Adam', 'LBFGS']):
    sub = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    if len(sub) == 0:
        continue
    b = best_row(sub)
    m, s = mean_std(b, 'train')
    et, _ = mean_std(b, 'epoch_times')
    tt = np.cumsum(et[:len(m)])
    plot_band(ax, tt, m, s, color=colors[i], lw=2, zorder=2, label=opt)

ax.set_xlabel('Wall Time (s)')
ax.set_ylabel('Train Loss')
ax.set_yscale('log')
ax.set_xscale('log')
plt.legend(ncol=2, loc='upper right', frameon=True, framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None, 2])
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_train_loss_best.pdf')
plt.show()

### Wall time vs val loss

In [ ]:
colors = sns.color_palette("deep")
fig, ax = plt.subplots(figsize=(8, 6))

best_svd_row = best_row(df_svd[df_svd['batch_size'] == bs])
val_mean, val_std = mean_std(best_svd_row, 'val')
et_mean, _ = mean_std(best_svd_row, 'epoch_times')
et_prepended = np.concatenate([[1.0], et_mean])
t = np.cumsum(et_prepended[:len(val_mean)])
plot_band(ax, t, val_mean, val_std, color='k', lw=3, zorder=3,
          label=f"Sven ($\\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)")

for i, opt in enumerate(['SGD', 'PolyakSGD', 'RMSprop', 'Adam', 'LBFGS']):
    sub = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    if len(sub) == 0:
        continue
    b = best_row(sub)
    m, s = mean_std(b, 'val')
    et, _ = mean_std(b, 'epoch_times')
    et = np.concatenate([[1.0], et])
    tt = np.cumsum(et[:len(m)])
    plot_band(ax, tt, m, s, color=colors[i], lw=2, zorder=2, label=opt)

ax.set_xlabel('Wall Time + 1 (sec)')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
ax.set_xscale('log')
plt.legend(ncol=2, loc='upper right', frameon=True, framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None, 2])
plt.xlim([1, None])
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_val_loss_best.pdf')
plt.show()

### Overlay different $k$ values: train loss

In [ ]:
for LR in [0.05, 0.1, 0.5]:
    for RTOL in [1e-4, 1e-3, 1e-2]:
        sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        if len(sel) == 0:
            continue
        k_values = sorted(sel['k'].unique())
        k_colors = sns.color_palette('viridis', n_colors=len(k_values))
        fig, ax = plt.subplots(figsize=(8, 6))
        n_ep = None
        for i, k in enumerate(k_values):
            row = sel[sel['k'] == k].iloc[0]
            m, s = mean_std(row, 'train')
            if n_ep is None:
                n_ep = len(m)
            epochs = np.arange(1, len(m) + 1)
            plot_band(ax, epochs, m, s, color=k_colors[i], lw=3, zorder=2, label=f"$k={int(k)}$")

        df_sgd = df_baseline[(df_baseline['batch_size'] == bs) & (df_baseline['optimizer'] == 'SGD')]
        if len(df_sgd) > 0:
            b = best_row(df_sgd)
            m, s = mean_std(b, 'train')
            epochs = np.arange(1, len(m) + 1)
            plot_band(ax, epochs, m, s, color='k', lw=3, zorder=3, ls='--',
                      label=f"SGD ($\\eta={lr_labels[b['lr']]}$)")

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Train Loss')
        ax.set_yscale('log')
        plt.legend(ncol=3, loc='upper center', frameon=True, framealpha=1)
        ax.set_title(f"Sven, Random Polynomial ($B = {bs}$, $\\eta={LR}$, rtol = ${lr_labels[RTOL]}$)")
        plt.ylim([None, 3])
        plt.xlim(0, (n_ep or 0) + 1)
        plt.xticks(np.arange(0, (n_ep or 0) + 1, max(1, (n_ep or 10) // 10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"train_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

### Overlay different $k$ values: val loss

In [ ]:
for LR in [0.05, 0.1, 0.5]:
    for RTOL in [1e-4, 1e-3, 1e-2]:
        sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        if len(sel) == 0:
            continue
        k_values = sorted(sel['k'].unique())
        k_colors = sns.color_palette('viridis', n_colors=len(k_values))
        fig, ax = plt.subplots(figsize=(8, 6))
        n_val = None
        for i, k in enumerate(k_values):
            row = sel[sel['k'] == k].iloc[0]
            m, s = mean_std(row, 'val')
            if n_val is None:
                n_val = len(m)
            epochs = np.arange(len(m))
            plot_band(ax, epochs, m, s, color=k_colors[i], lw=3, zorder=2, label=f"$k={int(k)}$")

        df_sgd = df_baseline[(df_baseline['batch_size'] == bs) & (df_baseline['optimizer'] == 'SGD')]
        if len(df_sgd) > 0:
            b = best_row(df_sgd)
            m, s = mean_std(b, 'val')
            epochs = np.arange(len(m))
            plot_band(ax, epochs, m, s, color='k', lw=3, zorder=3, ls='--',
                      label=f"SGD ($\\eta={lr_labels[b['lr']]}$)")

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation Loss')
        ax.set_yscale('log')
        plt.legend(ncol=3, loc='upper center', frameon=True, framealpha=1)
        ax.set_title(f"Sven, Polynomial ($B = {bs}$, $\\eta={LR}$, rtol = ${lr_labels[RTOL]}$)")
        plt.ylim([None, 3])
        plt.xlim(0, (n_val or 0))
        plt.xticks(np.arange(0, (n_val or 0) + 1, max(1, (n_val or 10) // 10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"val_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

### Number of nonzero SVs over training, per (lr, rtol)

In [ ]:
for LR in [0.05, 0.1, 0.5]:
    for RTOL in [1e-4, 1e-3, 1e-2]:
        sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        if len(sel) == 0:
            continue
        k_values = sorted(sel['k'].unique())
        k_colors = sns.color_palette('viridis', n_colors=len(k_values))
        fig, ax = plt.subplots(figsize=(8, 6))
        for i, k in enumerate(k_values):
            row = sel[sel['k'] == k].iloc[0]
            if row.get('svd_info') is None:
                continue
            m, s = mean_std(row, 'num_nonzero_svs', container='svd_info')
            # num_nonzero_svs is per-batch; reshape to per-epoch mean
            n_epoch = len(np.asarray(row['losses']['train']))
            if len(m) % n_epoch == 0:
                m = m.reshape(n_epoch, -1).mean(axis=1)
                s = s.reshape(n_epoch, -1).mean(axis=1)
            epochs = np.arange(1, len(m) + 1)
            plot_band(ax, epochs, m, s, color=k_colors[i], lw=2, zorder=2, label=f"$k={int(k)}$")

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Number of SVs Used by Sven')
        plt.legend(ncol=3, loc='upper center', frameon=True, framealpha=1)
        ax.set_title(f"Sven, Polynomial ($B = {bs}$, $\\eta={LR}$, rtol = ${lr_labels[RTOL]}$)")
        plt.grid(axis='both', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"num_nonzero_svs_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

### Number of nonzero SVs: rtol sweep at fixed k

In [ ]:
for LR in [0.05, 0.1, 0.5]:
    fig, ax = plt.subplots(figsize=(8, 6))
    K = 32
    styles = ['-', '--', ':']
    rtol_values = [1e-4, 1e-3, 1e-2]
    deep = sns.color_palette('deep')
    for j, RTOL in enumerate(rtol_values):
        sub = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL) & (df_svd['k'] == K)]
        if len(sub) == 0:
            continue
        row = sub.iloc[0]
        if row.get('svd_info') is None:
            continue
        m, s = mean_std(row, 'num_nonzero_svs', container='svd_info')
        n_epoch = len(np.asarray(row['losses']['train']))
        if len(m) % n_epoch == 0:
            m = m.reshape(n_epoch, -1).mean(axis=1)
            s = s.reshape(n_epoch, -1).mean(axis=1)
        epochs = np.arange(1, len(m) + 1)
        plot_band(ax, epochs, m, s, color=deep[0], lw=3, ls=styles[j],
                  label=f"$k={K}$, rtol = ${lr_labels[RTOL]}$")

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Number of SVs Used by Sven')
    plt.legend(ncol=1, loc='upper right', frameon=True, framealpha=1)
    ax.set_title(f"Sven, Polynomial ($B = {bs}$, $\\eta={LR}$)")
    plt.ylim([0, 35])
    plt.grid(axis='both', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"num_nonzero_svs_rtol_sweep_lr{LR}.pdf")
    plt.show()